## Tech Challenge Fase 4 — Análise de Vídeo Especializada para Saúde da Mulher

Sistema modular de monitoramento contínuo de vídeos clínicos, desenvolvido para o **Tech Challenge Fase 4**. Processa vídeos de cirurgias e procedimentos ginecológicos para identificar sinais precoces de risco à saúde e segurança feminina.

---

### Funcionalidades Implementadas

| # | Funcionalidade | Implementação |
|:---:|---|---|
| 1 | **Analisar vídeos de cirurgias ginecológicas** | 3 modelos YOLOv8 independentes treinados com datasets especializados (GynSurge, WCEBleedGen) |
| 2 | **Detecção de anomalias em tempo real** | Pipeline de 7 filtros por frame + classificação de severidade (MÉDIO / ALTO / CRÍTICO) |

---

### Modelos

| Modelo | Comportamento | Classes Detectadas | Critério Clínico |
|---|---|---|---|
| `instrumentos` | Rastreamento — identifica instrumentos utilizados | Pinça Grasper, Tesoura, Pinça Bipolar, Gancho | Sinais de complicações em cirurgias ginecológicas |
| `areas-criticas` | Anomalias de estruturas anatômicas | Útero, Tuba Uterina, Ovário | Desvios em procedimentos obstétricos |
| `sangramento` | Anomalias de sangramento cirúrgico | Sangramento em campo cirúrgico/endoscópico | Complicações obstétricas e trauma cirúrgico |

### Datasets Utilizados

| Modelo | Dataset | Fonte | Endereço |
|---|---|---|---|
| `instrumentos` | GynSurge — Instrument Segmentation | ITEC / Univ. Klagenfurt | https://ftp.itec.aau.at/datasets/GynSurge/ |
| `areas-criticas` | GynSurge — Anatomy Segmentation | ITEC / Univ. Klagenfurt | https://ftp.itec.aau.at/datasets/GynSurge/ |
| `sangramento` | WCEBleedGen (Cápsula Endoscópica) | Kaggle | https://www.kaggle.com/datasets/darksoul007fedsdfds/wcebleedgen |

In [1]:
# =====================================================================
#  INSTALAÇÃO DE DEPENDÊNCIAS
# =====================================================================
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q ultralytics==8.0.149 opencv-python-headless openai matplotlib seaborn scikit-learn scikit-image pandas
!pip install -q yt-dlp

ERROR: Could not find a version that satisfies the requirement torchaudio (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torchaudio

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =====================================================================
#  IMPORTS
# =====================================================================
import base64
import importlib.util
import os
import shutil
import subprocess
import sys
from datetime import datetime

from IPython.display import HTML, Markdown, display

### Configurações do Notebook (Vídeo e Aúdio)

Aqui podemos tanto baixar um vídeo do youtube para a análise quando utilizar algum vídeo local.

Aqui também definimos o tipo de análise do aúdio pelo parâmetro TIPO_CONSULTA.

Sendo as 4 opções:

- ginecologia
- pre_natal
- pos_parto
- violencia

Estaremos utilizando vídeos encontrados aqui:
https://www.laparoscopyhospital.com/Free_laparoscopic_gynecological_videos.htm

Executar python download_videos_teste.py para download dos vídeos de exemplo


In [3]:
# =====================================================================
#  CONFIGURAÇÕES DE VÍDEO E ANÁLISE
# =====================================================================

# --- FONTE DO VÍDEO (escolha uma das opções abaixo) ---

# Opção 1: URL do YouTube (deixe vazio "" para usar vídeo local)
YOUTUBE_URL = ""

# Opção 2: Caminho para vídeo local (deixe vazio "" para baixar do YouTube)
#   Exemplos:
#     VIDEO_LOCAL = "video_test.mp4"           # arquivo na raiz do projeto
#     VIDEO_LOCAL = r"C:\Videos\cirurgia.mp4"  # caminho absoluto
VIDEO_LOCAL = "videos/video_05.mp4"

# Nome do arquivo de trabalho quando baixado do YouTube
VIDEO_FILENAME = "videos/video_validacao.mp4"

# --- ANÁLISE DE ÁUDIO ---

# Tipo de consulta para contextualizar a análise clínica do áudio extraído do vídeo.
# Opções: "ginecologica" | "pre_natal" | "pos_parto" | "violencia"
TIPO_CONSULTA = "ginecologica"

### Configurações dos modelos e detectores

Nesta fase, utilizaremos os modelos pré-treinados disponíveis, sem realizar treinamento adicional. As configurações de treinamento estão presentes no arquivo `.env` para referência futura, mas não serão aplicadas nesta etapa.

In [4]:
# =====================================================================
#  CONFIGURAÇÃO
# =====================================================================

def _find_project_root():
    try:
        return os.path.dirname(os.path.abspath(__vsc_ipynb_file__))  # noqa: F821
    except NameError:
        pass
    for candidate in [os.path.abspath("."), os.path.dirname(os.path.abspath("."))]:
        if os.path.exists(os.path.join(candidate, "src", "relatorio.py")):
            return candidate
    return os.path.abspath(".")

PROJECT_ROOT = _find_project_root()
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(PROJECT_ROOT, ".env"))
except ImportError:
    pass

_DETECTORS_CONFIG = {
    "instrumentos":   ("src/detectors/instrumentos.py",   "InstrumentosDetector",  "instrumentos"),
    "areas-criticas": ("src/detectors/areas_criticas.py", "AreasCriticasDetector", "areas_criticas"),
    "sangramento":    ("src/detectors/sangramento.py",    "SangramentoDetector",   "sangramento"),
}

# Executa os 3 modelos para o relatório consolidado do Tech Challenge
_MODES_TO_RUN = list(_DETECTORS_CONFIG.keys())

_text_filter = os.getenv("FILTER_TEXT_REGIONS", "true").lower() not in ("0", "false", "no", "off")

print(f"Raiz do notebook : {PROJECT_ROOT}")
print(f"Modelos a rodar  : {_MODES_TO_RUN}")
print(f"Filtro de texto  : {'ATIVO' if _text_filter else 'DESATIVADO'} (FILTER_TEXT_REGIONS)")

Raiz do notebook : f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4
Modelos a rodar  : ['instrumentos', 'areas-criticas', 'sangramento']
Filtro de texto  : ATIVO (FILTER_TEXT_REGIONS)


### Limpeza de Output

Aqui limpamos as pastas de saída antes de gerarmos novos relatórios e outputs.

In [5]:
# =====================================================================
#  LIMPEZA DAS SAÍDAS ANTERIORES
# =====================================================================
_saida_root = os.path.join(PROJECT_ROOT, "saida")
_cleaned = []

if os.path.exists(_saida_root):
    for item in os.listdir(_saida_root):
        item_path = os.path.join(_saida_root, item)
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
            _cleaned.append(f"saida/{item}/")
        elif os.path.isfile(item_path):
            os.remove(item_path)
            _cleaned.append(f"saida/{item}")

if _cleaned:
    print("Saídas anteriores removidas:")
    for c in _cleaned:
        print(f"  - {c}")
else:
    print("Nenhuma saída anterior encontrada.")

os.makedirs(_saida_root, exist_ok=True)

Saídas anteriores removidas:
  - saida/areas_criticas/
  - saida/audio/
  - saida/instrumentos/
  - saida/parecer_medico.txt
  - saida/relatorio_geral.html
  - saida/relatorio_geral.txt
  - saida/sangramento/


### Identificação do vídeo a ser processado

In [6]:
# =====================================================================
#  FONTE DO VÍDEO
# =====================================================================
_use_local = bool(VIDEO_LOCAL and VIDEO_LOCAL.strip())
_use_youtube = bool(YOUTUBE_URL and YOUTUBE_URL.strip())

if not _use_local and not _use_youtube:
    raise ValueError("Preencha VIDEO_LOCAL ou YOUTUBE_URL na célula de configuração.")
if _use_local and _use_youtube:
    print("Ambas as fontes preenchidas — usando vídeo local (VIDEO_LOCAL tem prioridade).")
    _use_youtube = False

if _use_local:
    # Resolve caminho: absoluto ou relativo à raiz do projeto
    _local_path = VIDEO_LOCAL.strip()
    if not os.path.isabs(_local_path):
        _local_path = os.path.join(PROJECT_ROOT, _local_path)
    if not os.path.exists(_local_path):
        raise FileNotFoundError(f"Vídeo local não encontrado: {_local_path}")
    VIDEO_PATH = _local_path
    size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024
    print(f"Usando vídeo local: {VIDEO_PATH} ({size_mb:.1f} MB)")

else:
    VIDEO_PATH = os.path.join(PROJECT_ROOT, VIDEO_FILENAME)
    if os.path.exists(VIDEO_PATH):
        os.remove(VIDEO_PATH)
    print(f"Baixando: {YOUTUBE_URL}")
    result = subprocess.run(
        [
            "yt-dlp",
            "-f", "bestvideo[ext=mp4][height<=720]+bestaudio[ext=m4a]/best[ext=mp4][height<=720]",
            "--merge-output-format", "mp4",
            "-o", VIDEO_PATH,
            YOUTUBE_URL,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("STDOUT:", result.stdout[-2000:])
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError("Falha no download. Verifique a URL e se yt-dlp está instalado.")
    size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024
    print(f"Concluído: {VIDEO_FILENAME} ({size_mb:.1f} MB)")

Usando vídeo local: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\videos/video_05.mp4 (44.1 MB)


### Verificação dos Modelos

Aqui verificamos os modelos de vídeo já gerados anteriormente.

(executar app.py train --mode <modo> para gerar os modelos)
Recomendado GPU com suporte a CUDA
Os modelos já treinados aqui foram rodados localmente numa GPU NVIDIA, usando Yolov8.

Os modos de treino disponíveis:
- instrumentos
- sangramento
- areas-criticas

Como os modelos já foram disponibilizados, não é necessário executar o treinamento para executar este notebook, mas por fins de documentação:

Antes de executar o treinamento, rode:
- python app.py download --mode audio (irá gerar o dataset de audios)
- python app.py download --mode instrumentos (irá baixar e preparar o dataset de instrumentos)
- python app.py download --mode sangramento (irá baixar e preparar o dataset de sangramento)
- python app.py download --mode areas-criticas (irá baixar e preparar o dataset de áreas criticas)

In [7]:
# =====================================================================
#  VERIFICAÇÃO DOS MODELOS GERADOS ANTERIORMENTE 
# =====================================================================
faltando = []
for mode in _MODES_TO_RUN:
    _, _, folder = _DETECTORS_CONFIG[mode]
    pt = os.path.join(PROJECT_ROOT, "model", folder, "weights", "best.pt")
    status = "✓" if os.path.exists(pt) else "AUSENTE"
    if status == "AUSENTE":
        faltando.append(mode)
    print(f"  [{status}] {mode}")

if faltando:
    print(f"\nAVISO: modelo(s) ausente(s): {faltando}")
    print("  Execute: python app.py train --mode <modo>")
else:
    print("\nTodos os modelos foram encontrados com sucesso.")

  [✓] instrumentos
  [✓] areas-criticas
  [✓] sangramento

Todos os modelos foram encontrados com sucesso.


### Execução dos detectores

Neste passo executamos os detectores de vídeo com os nossos modelos YoloV8 carregados em cima do vídeo informado na primeira etapa.

In [8]:
# =====================================================================
#  EXECUÇÃO DOS MODELOS DE DETECÇÃO
# =====================================================================
def _load_module(name, rel_path):
    path = os.path.join(PROJECT_ROOT, rel_path)
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

model_results = []
total = len(_MODES_TO_RUN)

for i, mode in enumerate(_MODES_TO_RUN, 1):
    print(f"\n{'='*60}")
    print(f"  [{i}/{total}] {mode}")
    print(f"{'='*60}")

    rel_path, cls_name, folder = _DETECTORS_CONFIG[mode]
    mod      = _load_module(mode, rel_path)
    detector = getattr(mod, cls_name)()
    result   = detector.detect_video(VIDEO_PATH, headless=True)

    if result is None:
        print(f"  AVISO: {mode} pulado (modelo ausente ou erro).")
        continue

    if mode == "instrumentos":
        timeline = result.get("instrument_timeline", {})
        detected = [n for n, info in timeline.items() if info["count"] > 0]
        print(f"  Frames: {result['frame_count']}  |  "
              f"Detecções: {result['detections_count']}  |  "
              f"Instrumentos detectados: {detected}")
    else:
        pct = len(result["anomalies"]) / max(result["frame_count"], 1) * 100
        print(f"  Frames: {result['frame_count']}  |  "
              f"Detecções: {result['detections_count']}  |  "
              f"Anomalias: {len(result['anomalies'])} ({pct:.1f}%)")

    model_results.append({
        "model_folder":        folder,
        "frame_count":         result["frame_count"],
        "detections":          result["detections_count"],
        "anomalies":           result["anomalies"],
        "fps":                 result["fps"],
        "class_summary":       result.get("class_summary", {}),
        "instrument_timeline": result.get("instrument_timeline", {}),
    })

print(f"\n{'='*60}")
print(f"  Concluído: {len(model_results)}/{total} modelo(s) processado(s)")
print(f"{'='*60}")


  [1/3] instrumentos


f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\.venv\Lib\site-packages\ultralytics\utils\checks.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


Usando modelo: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\model\instrumentos\weights\best.pt
Iniciando análise [instrument_detector] ...
Vídeo: video_05.mp4 | 854x480 @ 30.0fps
Relatório TXT salvo em: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\instrumentos\relatorio.txt
Relatório HTML salvo em: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\instrumentos\relatorio.html

=== RESULTADO FINAL [instrument_detector] ===
Frames analisados : 5749
Detecções totais  : 33
  Pinca Grasper: 31 frames (0.5%) — 10 segmento(s)
  Tesoura: não detectado
  Pinca Bipolar: 2 frames (0.0%) — 1 segmento(s)
  Gancho: não detectado

Arquivos gerados em: saida/instrumentos/
  Frames: 5749  |  Detecções: 33  |  Instrumentos detectados: ['Pinca Grasper', 'Pinca Bipolar']

  [2/3] areas-criticas
Usando modelo: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\model\areas_criticas\weights\best.pt

### Extração e análise de áudio
Nesta seção extraímos o aúdio do vídeo e realizamos a análise do mesmo via IA.

In [9]:
# =====================================================================
#  EXTRAÇÃO E ANÁLISE DE ÁUDIO
#  Requer: ffmpeg no PATH + OPENAI_API_KEY no .env
# =====================================================================
import json as _json

audio_result = None   # dict com transcricao, nivel_risco, sinais_detectados, recomendacoes
AUDIO_PATH   = None   # caminho do MP3 extraído

# ── 1. Extrair áudio do vídeo ─────────────────────────────────────────────────
_audios_dir = os.path.join(PROJECT_ROOT, "audios")
os.makedirs(_audios_dir, exist_ok=True)

_base      = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
AUDIO_PATH = os.path.join(_audios_dir, f"{_base}.mp3")

_ffmpeg = shutil.which("ffmpeg")
if not _ffmpeg:
    print("AVISO: ffmpeg não encontrado no PATH — extração de áudio ignorada.")
    print("       Instale ffmpeg e adicione ao PATH para habilitar a análise de áudio.")
    AUDIO_PATH = None
else:
    print(f"Extraindo áudio de: {os.path.basename(VIDEO_PATH)}")
    _r = subprocess.run(
        [_ffmpeg, "-y", "-i", VIDEO_PATH, "-vn", "-acodec", "mp3", "-q:a", "2", AUDIO_PATH],
        capture_output=True, text=True,
    )
    if _r.returncode != 0:
        print(f"AVISO: Falha na extração de áudio:\n{_r.stderr[-500:]}")
        AUDIO_PATH = None
    else:
        _size_mb = os.path.getsize(AUDIO_PATH) / 1024 / 1024
        print(f"Áudio extraído: audios/{os.path.basename(AUDIO_PATH)} ({_size_mb:.1f} MB)")

# ── 2. Analisar áudio ─────────────────────────────────────────────────────────
if AUDIO_PATH and os.path.exists(AUDIO_PATH):
    _api_key = os.getenv("OPENAI_API_KEY")
    if not _api_key:
        print("AVISO: OPENAI_API_KEY não definida no .env — análise de áudio ignorada.")
        print("       Adicione OPENAI_API_KEY=sk-... ao arquivo .env para habilitar.")
    else:
        try:
            _audio_dir = os.path.join(PROJECT_ROOT, "audio_analisador")
            if _audio_dir not in sys.path:
                sys.path.insert(0, _audio_dir)

            _analyzer_mod = _load_module("analyzer", "audio_analisador/analyzer.py")
            _analyzer     = _analyzer_mod.AudioAnalyzer()

            print(f"Analisando áudio — tipo: {TIPO_CONSULTA} (usa API OpenAI, pode demorar)...")
            audio_result = _analyzer.analyze(AUDIO_PATH, TIPO_CONSULTA)

            # ── Salvar relatório de áudio ─────────────────────────────────────
            _audio_saida = os.path.join(PROJECT_ROOT, "saida", "audio")
            os.makedirs(_audio_saida, exist_ok=True)

            _audio_json = os.path.join(_audio_saida, "relatorio_audio.json")
            with open(_audio_json, "w", encoding="utf-8") as _f:
                _json.dump({
                    "tipo_consulta":    TIPO_CONSULTA,
                    "audio_arquivo":    os.path.basename(AUDIO_PATH),
                    "nivel_risco":      audio_result.get("nivel_risco", "—"),
                    "sinais_detectados": audio_result.get("sinais_detectados", "—"),
                    "recomendacoes":    audio_result.get("recomendacoes", "—"),
                    "transcricao":      audio_result.get("transcricao", "—"),
                }, _f, ensure_ascii=False, indent=2)

            _audio_txt = os.path.join(_audio_saida, "relatorio_audio.txt")
            with open(_audio_txt, "w", encoding="utf-8") as _f:
                _f.write("RELATÓRIO DE ANÁLISE CLÍNICA DE ÁUDIO\n")
                _f.write("=" * 60 + "\n")
                _f.write(f"Tipo de consulta : {TIPO_CONSULTA}\n")
                _f.write(f"Arquivo de áudio : {os.path.basename(AUDIO_PATH)}\n")
                _f.write(f"Nível de risco   : {audio_result.get('nivel_risco', '—')}\n\n")
                _f.write(f"TRANSCRIÇÃO:\n{audio_result.get('transcricao', '—')}\n\n")
                _f.write(f"SINAIS DETECTADOS:\n{audio_result.get('sinais_detectados', '—')}\n\n")
                _f.write(f"RECOMENDAÇÕES:\n{audio_result.get('recomendacoes', '—')}\n")

            print(f"Análise concluída | Risco: {audio_result.get('nivel_risco', '—')}")
            print(f"Relatório salvo em: saida/audio/")

        except ImportError as e:
            print(f"AVISO: Dependência ausente ({e})")
            print("       Execute: pip install openai librosa soundfile")
        except Exception as e:
            print(f"AVISO: Falha na análise de áudio: {e}")
else:
    print("Análise de áudio não realizada.")

Extraindo áudio de: video_05.mp4
Áudio extraído: audios/video_05.mp3 (3.0 MB)
Analisando áudio — tipo: ginecologica (usa API OpenAI, pode demorar)...


f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Análise concluída | Risco: MÉDIO
Relatório salvo em: saida/audio/


### Extração de Relatório
Nesta seção realizamos a construção do relatório consolidado, com os dados extraídos de todos os detectores e da análise de aúdio.

In [10]:
# =====================================================================
#  RELATÓRIO CONSOLIDADO
# =====================================================================
import relatorio as _rel
from IPython.display import Markdown, display
from datetime import datetime


def _ts(frame, fps):
    if frame is None:
        return "—"
    total = int(frame) // max(int(fps), 1)
    return f"{total // 60:02d}:{total % 60:02d}"


def _render_combined(model_results, video_path, fps, audio_result=None):
    vname  = os.path.basename(video_path) if video_path else "N/A"
    now    = datetime.now().strftime("%d/%m/%Y %H:%M")
    frames = model_results[0]["frame_count"] if model_results else 0
    total_anomalies = sum(len(r["anomalies"]) for r in model_results)
    rate    = total_anomalies / frames * 100 if frames else 0
    risk, _ = _rel._get_risk_level(rate)
    duration = _ts(frames, fps)

    md = []
    md.append("# Relatório Consolidado - Análise de Saúde da Mulher")
    md.append("**Análise de Vídeo Especializada para Saúde da Mulher**")
    md.append("")

    # ── Visão Geral ──────────────────────────────────────────────────────────
    md.append("## Visão Geral")
    md.append("| Frames Analisados | Modelos | Anomalias Totais | Taxa Geral | Risco |")
    md.append("|---:|:---:|---:|---:|:---:|")
    md.append(f"| {frames} | {len(model_results)} | {total_anomalies} | {rate:.1f}% | **{risk}** |")
    md.append("")

    for i, ev in enumerate(_rel._evaluate_criteria(model_results), 1):
        c      = ev["criterion"]
        status = "**[ACIONADO]**" if ev["triggered"] else "_Sem Ocorrências_"
        md.append(f"### Objetivo {i} — {c['title']}  {status}")
        if c.get("objective"):
            md.append(f"_{c['objective']}_")
            md.append("")
        md.append(f"**Escopo:** {c['description']}")
        md.append("")
        if ev["triggered"]:
            md.append("| Modelo | Ocorrências | Crítico | Alto | Médio |")
            md.append("|---|:---:|:---:|:---:|:---:|")
            for fd in ev["findings"]:
                md.append(f"| {fd['folder']} | {fd['count']} | {fd['critico']} | {fd['alto']} | {fd['medio']} |")
            md.append("")
            md.append(f"> **Recomendação:** {c['recommendation']}")
        else:
            md.append("> Nenhuma ocorrência detectada neste objetivo durante o procedimento analisado.")
        md.append("")

    # ── Instrumentos Cirúrgicos Utilizados ───────────────────────────────────
    for r in model_results:
        if r["model_folder"] == "instrumentos" and r.get("instrument_timeline"):
            md.append("## Instrumentos Cirúrgicos Utilizados")
            md.append("> Modo rastreamento — sem classificação de anomalias.")
            md.append("")
            md.append("| Instrumento | Detecções | % do Vídeo | Segmentos | Primeiro | Último |")
            md.append("|---|---:|---:|:---:|:---:|:---:|")
            for name, info in sorted(r["instrument_timeline"].items(), key=lambda x: -x[1]["count"]):
                if info["count"]:
                    n_seg = len(info["segments"])
                    md.append(
                        f"| {name} | {info['count']} | {info['frames_pct']:.1f}% "
                        f"| {n_seg} | {_ts(info['first_frame'], fps)} | {_ts(info['last_frame'], fps)} |"
                    )
                else:
                    md.append(f"| {name} | — | — | — | — | — |")
            md.append("")
            break

    # ── Resultados por Modelo ────────────────────────────────────────────────
    md.append("## Resultados por Modelo")
    md.append("| Modelo | Detecções | Anomalias | Taxa | Risco |")
    md.append("|---|:---:|:---:|:---:|:---:|")
    for r in model_results:
        arate = len(r["anomalies"]) / r["frame_count"] * 100 if r["frame_count"] else 0
        if r["model_folder"] == "instrumentos":
            md.append(f"| {r['model_folder']} | {r['detections']} | — | — | rastreamento |")
        else:
            mlabel, _ = _rel._get_risk_level(arate)
            md.append(f"| {r['model_folder']} | {r['detections']} | {len(r['anomalies'])} | {arate:.1f}% | {mlabel} |")
    md.append("")

    # ── Linha do Tempo — Todas as Anomalias ──────────────────────────────────
    all_anomalies = [
        (r["model_folder"], a, r.get("fps", fps))
        for r in model_results
        for a in r["anomalies"]
        if isinstance(a, dict)
    ]
    md.append("## Linha do Tempo — Todas as Anomalias")
    if not all_anomalies:
        md.append("_Nenhuma anomalia detectada em nenhum modelo._")
    else:
        md.append("| Modelo | Timestamp | Frame | Severidade | Tipo | Descrição |")
        md.append("|:---:|:---:|:---:|:---:|:---:|---|")
        for folder, a, fps_r in sorted(all_anomalies, key=lambda x: x[1].get("frame", 0)):
            f   = a.get("frame", 0)
            sev = a.get("severity", "-")
            md.append(
                f"| {folder} | {_ts(f, fps_r)} | {f} | {sev} "
                f"| {a.get('type','-')} | {a.get('description','')} |"
            )

    # ── Análise Clínica de Áudio ──────────────────────────────────────────────
    md.append("")
    md.append("## Análise Clínica de Áudio")
    if audio_result is None:
        md.append("_Análise de áudio não realizada (ffmpeg ou OPENAI_API_KEY não disponíveis)._")
    else:
        nivel = audio_result.get("nivel_risco", "—")
        _risk_badge = {"ALTO": "**ALTO**", "MÉDIO": "**MÉDIO**", "BAIXO": "**BAIXO**"}.get(nivel, nivel)
        md.append("| Campo | Valor |")
        md.append("|---|---|")
        md.append(f"| **Tipo de Consulta** | {TIPO_CONSULTA.replace('_', ' ').title()} |")
        md.append(f"| **Nível de Risco** | {_risk_badge} |")
        md.append("")
        md.append("**Transcrição:**")
        md.append(f"> {audio_result.get('transcricao', '—')}")
        md.append("")
        md.append("**Sinais Detectados:**")
        md.append(f"> {audio_result.get('sinais_detectados', '—')}")
        md.append("")
        md.append("**Recomendações:**")
        md.append(f"> {audio_result.get('recomendacoes', '—')}")

    return "\n".join(md)


# ── Executar ──────────────────────────────────────────────────────────────────

if not model_results:
    display(Markdown("### Nenhum resultado disponível para gerar relatório."))
else:
    fps       = model_results[0].get("fps", 20)
    saida_dir = os.path.join(PROJECT_ROOT, "saida")
    os.makedirs(saida_dir, exist_ok=True)

    report_txt = os.path.join(saida_dir, "relatorio_geral.txt")
    _rel.generate_combined_report(report_txt, model_results, VIDEO_PATH)

    display(Markdown(_render_combined(model_results, VIDEO_PATH, fps, audio_result=audio_result)))

Relatório consolidado TXT: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\relatorio_geral.txt
Relatório consolidado HTML: f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\relatorio_geral.html


# Relatório Consolidado - Análise de Saúde da Mulher
**Análise de Vídeo Especializada para Saúde da Mulher**

## Visão Geral
| Frames Analisados | Modelos | Anomalias Totais | Taxa Geral | Risco |
|---:|:---:|---:|---:|:---:|
| 5749 | 3 | 111 | 1.9% | **BAIXO** |

### Objetivo 1 — Riscos em Saúde Materna e Ginecológica  **[ACIONADO]**
_Detectar precocemente riscos em saúde materna e ginecológica_

**Escopo:** Monitoramento de estruturas ginecológicas (útero, tuba uterina, ovário) e sangramento anômalo durante procedimentos obstétricos e cirúrgicos.

| Modelo | Ocorrências | Crítico | Alto | Médio |
|---|:---:|:---:|:---:|:---:|
| areas_criticas | 111 | 34 | 77 | 0 |

> **Recomendação:** Acionar equipe obstétrica e ginecológica para avaliação imediata das anomalias detectadas.

### Objetivo 2 — Bem-Estar Psicológico Feminino  _Sem Ocorrências_
_Monitorar bem-estar psicológico feminino_

**Escopo:** Monitoramento de eventos hemorrágicos e complicações cirúrgicas que podem impactar negativamente o bem-estar e a recuperação psicológica pós-operatória da paciente.

> Nenhuma ocorrência detectada neste objetivo durante o procedimento analisado.

### Objetivo 3 — Detecção de Anomalias em Tempo Real  **[ACIONADO]**
_Aplicar técnicas de detecção de anomalias em tempo real para monitoramento preventivo específico_

**Escopo:** Monitoramento contínuo frame a frame com classificação automática de severidade (MÉDIO / ALTO / CRÍTICO): ausências de estruturas anatômicas, sangramentos persistentes e variações bruscas de campo cirúrgico.

| Modelo | Ocorrências | Crítico | Alto | Médio |
|---|:---:|:---:|:---:|:---:|
| areas_criticas | 111 | 34 | 77 | 0 |

> **Recomendação:** Revisar os segmentos sinalizados e acionar protocolo de monitoramento preventivo.

## Instrumentos Cirúrgicos Utilizados
> Modo rastreamento — sem classificação de anomalias.

| Instrumento | Detecções | % do Vídeo | Segmentos | Primeiro | Último |
|---|---:|---:|:---:|:---:|:---:|
| Pinca Grasper | 31 | 0.5% | 10 | 00:02 | 02:36 |
| Pinca Bipolar | 2 | 0.0% | 1 | 02:06 | 02:06 |
| Tesoura | — | — | — | — | — |
| Gancho | — | — | — | — | — |

## Resultados por Modelo
| Modelo | Detecções | Anomalias | Taxa | Risco |
|---|:---:|:---:|:---:|:---:|
| instrumentos | 33 | — | — | rastreamento |
| areas_criticas | 1554 | 111 | 1.9% | BAIXO |
| sangramento | 0 | 0 | 0.0% | BAIXO |

## Linha do Tempo — Todas as Anomalias
| Modelo | Timestamp | Frame | Severidade | Tipo | Descrição |
|:---:|:---:|:---:|:---:|:---:|---|
| areas_criticas | 00:00 | 10 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:01 | 30 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:01 | 41 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:03 | 102 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:04 | 122 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:11 | 345 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:12 | 365 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:19 | 570 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:19 | 590 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:25 | 777 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:26 | 797 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:27 | 832 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:28 | 852 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:32 | 976 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:33 | 996 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:34 | 1041 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:38 | 1169 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:40 | 1207 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:40 | 1220 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:41 | 1240 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:42 | 1274 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:43 | 1292 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:43 | 1312 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:44 | 1332 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:45 | 1352 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:47 | 1435 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:48 | 1454 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:50 | 1500 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:50 | 1528 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:51 | 1548 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:53 | 1619 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:54 | 1639 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 00:55 | 1674 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 00:56 | 1694 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:03 | 1893 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:06 | 1988 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:06 | 2008 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:08 | 2066 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:09 | 2086 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:10 | 2122 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:11 | 2141 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:12 | 2161 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:18 | 2369 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:19 | 2389 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:20 | 2415 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:21 | 2435 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:26 | 2584 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:26 | 2604 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:27 | 2635 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:29 | 2696 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:30 | 2716 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:31 | 2746 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:32 | 2766 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:32 | 2786 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:34 | 2825 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:34 | 2845 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:35 | 2879 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:36 | 2899 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:38 | 2950 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:39 | 2994 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:40 | 3017 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:41 | 3037 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:41 | 3052 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:42 | 3072 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:49 | 3293 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:50 | 3316 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:50 | 3327 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:52 | 3362 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:52 | 3382 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 01:53 | 3413 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 01:54 | 3433 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 02:02 | 3665 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:03 | 3705 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:05 | 3768 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:06 | 3788 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:06 | 3808 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 02:07 | 3828 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:09 | 3875 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:10 | 3904 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:14 | 4038 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:19 | 4187 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:20 | 4207 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 02:22 | 4266 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:23 | 4315 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:24 | 4348 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:25 | 4371 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:27 | 4424 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:32 | 4562 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:35 | 4660 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:37 | 4728 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:38 | 4746 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:40 | 4807 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:40 | 4827 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 02:42 | 4862 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:43 | 4891 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:43 | 4911 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 02:45 | 4958 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:45 | 4975 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:47 | 5032 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:48 | 5052 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 02:55 | 5266 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:56 | 5295 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:56 | 5309 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:57 | 5326 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:57 | 5339 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:59 | 5379 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 02:59 | 5399 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |
| areas_criticas | 03:04 | 5532 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 03:06 | 5590 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 03:06 | 5601 | ALTO | AUSÊNCIA | Estruturas ginecológicas ausentes (10 frames) — verificar campo cirúrgico e protocolo obstétrico |
| areas_criticas | 03:07 | 5621 | CRÍTICO | AUSÊNCIA | Estruturas ginecológicas não identificadas (30 frames) — possível desvio de procedimento obstétrico ou complicação grave |

## Análise Clínica de Áudio
| Campo | Valor |
|---|---|
| **Tipo de Consulta** | Ginecologica |
| **Nível de Risco** | **MÉDIO** |

**Transcrição:**
> Bom dia, este é um caso de diagnóstico de laparoscópio, teste de potência tubal e drilagem ovariana para doença de PCOD. Aqui a uterus parece normal, o apêndice recidivina coli também parece absolutamente normal desta vez. Aqui a ponta do apêndice, isto é normal. Agora você pode ver o triângulo do domo, isto é também normal. Aqui é a uterus, que parece absolutamente normal. Agora o apêndice será injetado pela canula HSG do cerveja, e temos que verificar, aqui o apêndice está vindo do lado direito, isto é absolutamente normal. Agora este é o lado esquerdo do apêndice, só estamos levantando o apêndice, para que possamos ver o apêndice vindo, e aqui, o lado esquerdo também está bem, o apêndice azul metileno estava vindo. Agora este paciente, no ultrassom, foi diagnosticado que ela tem síndrome de laparoscópio, e ela estava tendo o nível de testosterona também alto, então nós estamos apenas fazendo o abertura do ovário, e algumas das pontas serão feitas, durante o abertura do ovário, você tem que usar somente uma corrente de 40W, se você está usando uma cirurgia eletrônica, e o máximo de 4 a 8 pontas devem ser feitas, para que o córtex do ovário não seja muito danificado. A vantagem é que o paciente vai ter uma redução fácil no nível do hormônio masculino e do hormônio. É especialmente uma boa procedida para o paciente que teve a terapia médica, e agora mais uma abertura será feita, então esta abertura será feita no lado direito, agora nós faremos no lado esquerdo, este é o ovário esquerdo, e ela terá o ovário bilateral policístico, este é o abertura com a corrente de corte monopolar, e apenas algumas aberturas serão feitas, e agora esta procedida será completada. Então, 4 aberturas foram feitas, agora mais algumas no lado esquerdo, este é um fio PCOE, que é especialmente um fio de design, que pode cortar apenas 4 milímetros, e agora a abertura do ovário está completa. Então, muito obrigado por assistir este vídeo, esta é uma procedida muito simples, obrigado. Legendas pela comunidade Amara.org


**Sinais Detectados:**
> A paciente apresenta energia vocal muito baixa e voz grave, o que pode indicar exaustão extrema ou depressão severa. Há uma alta taxa de pausas longas, com uma pausa particularmente longa de 6.5s, sugerindo hesitação ou bloqueio emocional. Apesar do conteúdo técnico e neutro da transcrição, o padrão acústico sugere um estado emocional de exaustão ou dissociação. Não há menção direta a violência, mas a dissociação pode ser um indicador velado de trauma.

**Recomendações:**
> Recomenda-se uma avaliação mais aprofundada do estado emocional da paciente, possivelmente envolvendo um psicólogo ou psiquiatra. Investigar possíveis causas de exaustão ou depressão, e considerar a possibilidade de trauma psicológico. Monitorar a paciente para sinais de violência doméstica ou sexual, mesmo que não explicitamente mencionados.

### Parecer médico via IA

Aqui coletamos os dados de todos os relatórios de todos os detectores e realizamos via I.A a geração do parecer técnico definitivo.

In [11]:
# =====================================================================
#  PARECER MÉDICO VIA IA (GPT-4o)
#  Envia os resultados de detecção para o GPT-4o e gera um parecer
#  médico completo e profissional. Requer OPENAI_API_KEY no .env
# =====================================================================

parecer_medico = None  # str com o parecer ou None se não disponível

_api_key = os.getenv("OPENAI_API_KEY")
if not _api_key:
    print("AVISO: OPENAI_API_KEY não definida — parecer médico via IA ignorado.")
    print("       Adicione OPENAI_API_KEY=sk-... ao arquivo .env para habilitar.")
elif not model_results:
    print("AVISO: sem resultados de detecção para gerar parecer.")
else:
    try:
        _opinion_dir = os.path.join(PROJECT_ROOT, "video_analisador")
        if _opinion_dir not in sys.path:
            sys.path.insert(0, _opinion_dir)

        from medical_opinion import generate_medical_opinion

        print(f"Gerando parecer médico via GPT-4o (pode demorar algum tempo)...")
        parecer_medico, _err = generate_medical_opinion(
            model_results, os.path.basename(VIDEO_PATH)
        )

        if _err:
            print(f"AVISO: {_err}")
            parecer_medico = None
        else:
            # Salvar parecer
            _parecer_saida = os.path.join(PROJECT_ROOT, "saida", "parecer_medico.txt")
            with open(_parecer_saida, "w", encoding="utf-8") as _f:
                _f.write(parecer_medico)
            print(f"Parecer médico salvo em: saida/parecer_medico.txt")

            # Exibir inline no notebook
            display(Markdown("---"))
            display(Markdown("## Parecer Médico — IA (GPT-4o)"))
            display(Markdown(parecer_medico))

    except ImportError as e:
        print(f"AVISO: Dependência ausente ({e}) — execute: pip install openai")
    except Exception as e:
        print(f"AVISO: Falha ao gerar parecer médico: {e}")

Gerando parecer médico via GPT-4o (pode demorar algum tempo)...
Parecer médico salvo em: saida/parecer_medico.txt


---

## Parecer Médico — IA (GPT-4o)

**1. Resumo Executivo**

A análise computacional do vídeo cirúrgico "video_05.mp4" revelou um total de 111 anomalias em 1.9% dos frames analisados. O sistema de visão computacional identificou a presença de instrumentos cirúrgicos, áreas críticas e não detectou eventos de sangramento. As anomalias foram predominantemente relacionadas à ausência de estruturas ginecológicas, com implicações potenciais para o procedimento.

**2. Análise dos Achados por Modelo de Detecção**

- **Instrumentos**: O modelo identificou a presença de instrumentos cirúrgicos, com 33 detecções totais. A pinça Grasper foi o instrumento mais frequentemente detectado, presente em 31 ocasiões ao longo do vídeo, enquanto a pinça bipolar foi identificada apenas duas vezes. Não foram detectadas anomalias relacionadas ao uso dos instrumentos, sugerindo um manejo adequado dos mesmos durante o procedimento.

- **Áreas Críticas**: Este modelo identificou 1554 detecções de estruturas ginecológicas, com 111 anomalias. As estruturas mais frequentemente identificadas foram a tuba uterina, o útero e o ovário. As anomalias detectadas foram classificadas como críticas ou de alto risco, principalmente devido à ausência de identificação de estruturas ginecológicas em momentos específicos do vídeo.

- **Sangramento**: Não foram detectados eventos de sangramento durante o procedimento, indicando que não houve complicações hemorrágicas significativas.

**3. Avaliação de Risco Clínico**

O nível de risco global do procedimento é considerado elevado devido à alta incidência de anomalias críticas relacionadas à ausência de estruturas ginecológicas. A ausência de identificação dessas estruturas pode indicar desvios no procedimento ou complicações graves, exigindo atenção imediata para garantir a segurança do paciente.

**4. Eventos e Anomalias Relevantes**

As anomalias mais significativas ocorreram nos primeiros minutos do vídeo, com repetidas detecções de ausência de estruturas ginecológicas. Estes eventos foram classificados como críticos e de alto risco, sugerindo possíveis desvios do protocolo cirúrgico ou complicações que necessitam de investigação adicional.

**5. Recomendações**

- Revisar o campo cirúrgico e o protocolo obstétrico para garantir a correta identificação das estruturas ginecológicas.
- Considerar a realização de uma revisão intraoperatória para confirmar a integridade e a presença das estruturas ginecológicas.
- Implementar protocolos de monitoramento contínuo para detectar e corrigir desvios no procedimento em tempo real.
- Avaliar a necessidade de treinamento adicional para a equipe cirúrgica em técnicas de visualização laparoscópica.

**6. Considerações Finais**

A análise sugere a necessidade de atenção redobrada à identificação das estruturas ginecológicas durante procedimentos laparoscópicos. Embora o sistema de visão computacional forneça suporte valioso, ele não substitui a avaliação clínica direta. Recomenda-se que a equipe médica valide os achados e tome decisões baseadas em julgamento clínico. Limitações do sistema incluem a possibilidade de falsos positivos ou negativos, que devem ser considerados na interpretação dos resultados. O acompanhamento contínuo do caso é essencial para garantir a segurança e eficácia do tratamento.

### Lista de Relatórios e Output gerado

Aqui só iremos listar aonde os relatórios e o video com output de resultado foram gerados.

In [12]:
# =====================================================================
#  ARQUIVOS GERADOS
# =====================================================================
from IPython.display import Markdown, display

if not model_results:
    display(Markdown("_Nenhum resultado disponível._"))
else:
    lines = ["## Arquivos Gerados", ""]

    # ── Vídeos anotados ───────────────────────────────────────────────────────
    lines.append("### Vídeos Anotados")
    lines.append("| Modelo | Caminho | Tamanho |")
    lines.append("|---|---|---:|")
    for r in model_results:
        folder   = r["model_folder"]
        vid_path = os.path.join(PROJECT_ROOT, "saida", folder, "resultado.mp4")
        if os.path.exists(vid_path):
            size_mb = os.path.getsize(vid_path) / 1024 / 1024
            lines.append(f"| `{folder}` | `{vid_path}` | {size_mb:.1f} MB |")
        else:
            lines.append(f"| `{folder}` | _não gerado_ | — |")

    # ── Relatórios e pareceres ────────────────────────────────────────────────
    lines.append("")
    lines.append("### Relatórios e Pareceres")
    lines.append("| Arquivo | Formato | Caminho |")
    lines.append("|---|:---:|---|")

    for ext in ("txt", "html"):
        rpath = os.path.join(PROJECT_ROOT, "saida", f"relatorio_geral.{ext}")
        if os.path.exists(rpath):
            lines.append(f"| Consolidado Tech Challenge | `{ext.upper()}` | `{rpath}` |")

    for r in model_results:
        folder = r["model_folder"]
        for ext in ("txt", "html"):
            rpath = os.path.join(PROJECT_ROOT, "saida", folder, f"relatorio.{ext}")
            if os.path.exists(rpath):
                lines.append(f"| {folder} | `{ext.upper()}` | `{rpath}` |")

    # Parecer médico consolidado
    for p, label in (
        (os.path.join(PROJECT_ROOT, "saida", "parecer_medico.txt"), "Parecer Médico IA (geral)"),
        *[
            (os.path.join(PROJECT_ROOT, "saida", r["model_folder"], "parecer_medico.txt"),
             f"Parecer Médico IA ({r['model_folder']})")
            for r in model_results
        ],
    ):
        if os.path.exists(p):
            lines.append(f"| {label} | `TXT` | `{p}` |")

    # Áudio
    rpath = os.path.join(PROJECT_ROOT, "saida", "audio", "relatorio_audio.txt")
    if os.path.exists(rpath):
        lines.append(f"| Áudio — Análise Clínica | `TXT` | `{rpath}` |")

    display(Markdown("\n".join(lines)))

## Arquivos Gerados

### Vídeos Anotados
| Modelo | Caminho | Tamanho |
|---|---|---:|
| `instrumentos` | _não gerado_ | — |
| `areas_criticas` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\areas_criticas\resultado.mp4` | 106.3 MB |
| `sangramento` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\resultado.mp4` | 99.1 MB |

### Relatórios e Pareceres
| Arquivo | Formato | Caminho |
|---|:---:|---|
| Consolidado Tech Challenge | `TXT` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\relatorio_geral.txt` |
| Consolidado Tech Challenge | `HTML` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\relatorio_geral.html` |
| instrumentos | `TXT` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\instrumentos\relatorio.txt` |
| instrumentos | `HTML` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\instrumentos\relatorio.html` |
| areas_criticas | `TXT` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\areas_criticas\relatorio.txt` |
| areas_criticas | `HTML` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\areas_criticas\relatorio.html` |
| sangramento | `TXT` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\relatorio.txt` |
| sangramento | `HTML` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\sangramento\relatorio.html` |
| Parecer Médico IA (geral) | `TXT` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\parecer_medico.txt` |
| Áudio — Análise Clínica | `TXT` | `f:\Projetos\TECHCHALLENGE_IADevs_2025\FelipeMoraes\Tech Challenge Fase 4\saida\audio\relatorio_audio.txt` |